In [ ]:
from typing import List, TypedDict
import re

from langchain_community.document_loaders import PyPDFLoader
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END
from pydantic import BaseModel
from dotenv import load_dotenv

load.dotenv()


ModuleNotFoundError: No module named 'langchain_community'

In [ ]:
# docs loading
docs = (
    PyPDFLoader("./documents/Project\RAG_Campus_X\Advance_RAG (CRAG)/book1.pdf").load()
    + PyPDFLoader("./documents/Project\RAG_Campus_X\Advance_RAG (CRAG)/book2.pdf").load()
    + PyPDFLoader("./documents/Project\RAG_Campus_X\Advance_RAG (CRAG)/book3.pdf").load()
)

In [ ]:
# text splitting
chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(docs)
for d in chunks:
  d.page_content = d.page_content.encode("utf-8", 'ignore').decode("utf-8", 'ignore')

In [ ]:
# embedding --> vectore store and created retriever

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vector_store = FAISS.from_documents(chunks, embeddings)
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
# craeating llm model
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

In [ ]:
# defining states (this is a langgraph process)

class State(TypedDict):
  question:str
  docs:List[Document]

  strips:List[str]          # output of decomposition (sentence strips)
  kept_strips:List[str]     # after filtering (kept sentences)
  refined_context:str       #recomposed internal knowledge (joined kept_strips)

  answer:str

In [ ]:
# retrieved node

def retrieved(state:State) -> State:
  q = state['question']
  return {'docs': retriever.invoke(q)}

In [ ]:
# this will be a refinement node

#---------------------
# Sentence-level Decomposition
#---------------------

# It will convert the given str into strips
def decompose_to_sentences(text: str) -> List[str]:
  text = re.sub(r"\s+", " ", text).strip()
  sentences = re.split(r"(?<=[.!?])\s+", text)
  return [s.strip() for s in sentences if len(s.strip()) > 20]

#---------------------
# Filter LLM(judge)
#---------------------

#Giving prompt to llm

class KeepOrDrop(BaseModel):
  keep: bool

filter_prompt = ChatPromptTemplate.from_messages(
     [
         (
             "system",
             "You are a strict relevance filter.\n"
             "Return keep=true only if sentence directly helps answer the question.\n"
             "Use only the sentence. Output JSON only."
         ),
         ("human", "Question: {question}\n\nSentence: {sentence}"),
     ]
 )

filter_chain = filter_prompt | llm.with_structured_output(KeepOrDrop)

#-----------------------
# REFINING (Decompose -> Filter -> Recompose)
#-----------------------

# this will be a refining node

def refine(state:State) -> State:

  q = state['question']

 # Combine retrieved doc into one context string
context = "\n\n".join(d.page_content for d in State['docs']).strip()

# 1) DECOMPOSITION : context -> sentence strips
strips = decompose_to_sentences(context)      # we are creating s strips here

# 2) FILTER: keep only relevant strips
kept: List[str] = []

for s in strips:
  if filter_chain.invoke({"question": q, "sentence": s})
      kept.append(s)

      # what we are doing here , we are picking every strips and sending it to filter_chain.
      # Then it will give que and strips to llm and asking llm that "is it right strip for that
      # ques and if yes then it adding to that kept wali line."

# 3) RECOMPOSE : glue kept strip back together (internal knowledge)

refined_context = "\n".join(kept).strip()    # kept strips storing in this variable

return {
    "strips": strips,
    "kept_strips": kept,
    "refined_context": refined_context

}

In [ ]:
answer_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are a helpful assistant. Answer only using the provided refined bullets.\n"
            "If the bullets are empty or insufficient, say: 'I don't know based on provided books.'",
        ),
    ]
)

def genrate(state:State) -> State:
  out = (answer_prompt | llm).invoke({"question": state['question'],"refined_context": state['refined_context']})
  return {"answer": out.content}

In [ ]:
g = StateGraph(State)
g.add_node("retrive", retrive)
g.add_node("refine", refine)
g.add_node("genrate", genrate)

g.add_edge(START, "retrive")
g.add_edge("retrive", "refine")
g.add_edge("refine", "genrate")
g.add_edge("genrate", END)

app = g.compile()
app

# here we are joining these three things 1)retrive, 2)refine, 3) generate

In [ ]:
res = app.invoke({
    "question": "What is nykaa advertising about?",
    "docs":[],
    "strips": [],
    "kept_strips":[],
    "refined_context":"",
    "answer":""
})
print(res['answer'])      # on the basis of strips answer will bbe ore pointwise


Till now we have completed pahse 1 - refined required context from the books accordingly to ques